<a href="https://colab.research.google.com/github/karsarobert/LLM2026/blob/main/ChatGpt2025_05_feladatokkal.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Chat GPT és más nagy nyelvi modellek alkalmazása
##PTE Gépi tanulás
###5. gyakorlat: JSON az LLM programozásban
2026. március 2.

**Miért kell nekünk a JSON?**

Az előző gyakorlaton láttuk, hogy a nagy nyelvi modellek (LLM-ek) képesek strukturálatlan szövegből (pl. egy beszkennelt számláról) adatokat kinyerni és azokat **JSON formátumban** visszaadni. Ez a képesség az LLM-alapú alkalmazások egyik leghasznosabb tulajdonsága.

De mi is pontosan a JSON, és hogyan tudunk vele hatékonyan dolgozni Python-ban? Ez a gyakorlat erre ad választ.

**A gyakorlaton tárgyalt témák:**

* Mi a JSON és mire használjuk?
* JSON szintaxis – az alapvető szabályok
* JSON kezelése Python-ban (a `json` modul)
* JSON az LLM programozásban – strukturált kimenet kérése
* Gemini API és JSON séma
* Feladatok és gyakorlatok

---
# 1. Mi a JSON?

A **JSON** (JavaScript Object Notation) egy könnyűsúlyú, ember által is olvasható adatcsere-formátum.  
Neve ellenére **nem csak JavaScript-ben** használják – szinte minden modern programozási nyelv ismeri (Python, Java, C#, PHP, stb.).

### Miért vált szabvánnyá?
- **Egyszerű** – kevés szabály, könnyen tanulható
- **Olvasható** – ember és gép számára egyaránt érthető
- **Univerzális** – webes API-k, adatbázisok, konfigurációs fájlok, LLM-ek mind használják
- **Hierarchikus** – jól leképezi az összetett adatstruktúrákat

### Összehasonlítás más formátumokkal

| Jellemző | JSON | XML | CSV |
|---|---|---|-|
| Olvashatóság | ✅ Kiváló | ⚠️ Verbose | ✅ Egyszerű |
| Hierarchia | ✅ Igen | ✅ Igen | ❌ Lapos |
| Adattípusok | ✅ Igen | ❌ Csak szöveg | ❌ Csak szöveg |
| Méret | ✅ Kisebb | ❌ Nagyobb | ✅ Legkisebb |
| LLM-ek által preferált | ✅ Igen | ⚠️ Ritkán | ❌ Nem |

**Az LLM-ek a JSON-t részesítik előnyben** – könnyebben generálható, validálható, és közvetlenül feldolgozható Python-ban.

---
# 2. JSON szintaxis – az alapvető szabályok

A JSON **hat** adattípust ismer:

In [ ]:
import json

# JSON típusok szemléltetése Python szótáron keresztül
# (A Python dict és a JSON objektum nagyon hasonló!)

pelda = {
    "nev": "Kovács Péter",          # string – idézőjelbe (!)
    "kor": 32,                       # integer
    "magassag": 1.82,                # float
    "aktiv": True,                   # boolean (JSON-ban: true/false)
    "kutya": None,                   # null (JSON-ban: null)
    "hobbik": ["futás", "olvasás", "programozás"],  # tömb (array)
    "cim": {                         # beágyazott objektum
        "varos": "Pécs",
        "iranyitoszam": "7621"
    }
}

# Python dict-ből JSON szöveg
szoveg = json.dumps(pelda, indent=2, ensure_ascii=False)
print(szoveg)

{
  "nev": "Kovács Péter",
  "kor": 32,
  "magassag": 1.82,
  "aktiv": true,
  "kutya": null,
  "hobbik": [
    "futás",
    "olvasás",
    "programozás"
  ],
  "cim": {
    "varos": "Pécs",
    "iranyitoszam": "7621"
  }
}


### ⚠️ Legfontosabb JSON-szintaxis szabályok

1. **Kulcsok mindig dupla idézőjelben** – `"nev"` ✅, `nev` ❌, `'nev'` ❌
2. **Boolean értékek JSON-ban kisbetűsen** – `true` / `false` (Pythonban: `True` / `False`)
3. **Null értékek** – `null` (JSON-ban), Pythonban `None`
4. **Nincs komment** – a JSON nem enged megjegyzéseket
5. **Az utolsó elem után nincs felesleges vessző**

```json
// HELYES JSON:
{
  "szamla_szam": "2024-001",
  "osszeg": 15000,
  "fizetett": false,
  "tetelek": [
    {"nev": "Laptop", "ar": 12000},
    {"nev": "Egér", "ar": 3000}
  ]
}
```

---
# 3. JSON kezelése Python-ban – a `json` modul

Python-ban a beépített `json` modul segítségével dolgozunk JSON adatokkal.  
**Nem kell telepíteni** – a Python standard könyvtárának része.

| Függvény | Irány | Leírás |
|---|---|---|
| `json.loads(szöveg)` | JSON string → Python dict | Szöveg értelmezése |
| `json.dumps(adat)` | Python dict → JSON string | Szöveggé alakítás |
| `json.load(fájl)` | JSON fájl → Python dict | Fájlból beolvasás |
| `json.dump(adat, fájl)` | Python dict → JSON fájl | Fájlba írás |

> **Emlékeztető:** az `s` betű a `loads`/`dumps`-ban a **string**-re utal.

In [ ]:
import json

# ─────────────────────────────────────────
# json.loads() – JSON szöveg → Python dict
# ─────────────────────────────────────────
# Képzeljük el, hogy az LLM ezt a szöveget adja vissza:
llm_valasz = """
{
  "szemely": "Nagy Anna",
  "erzelem": "pozitív",
  "pontszam": 0.92,
  "kulcsszavak": ["kiváló", "elégedett", "ajánlom"]
}
"""

adat = json.loads(llm_valasz)   # szövegből Python dict lesz
print(type(adat))               # <class 'dict'>
print(adat["szemely"])          # Nagy Anna
print(adat["kulcsszavak"])      # ['kiváló', ...]
print(adat["pontszam"])         # 0.92

<class 'dict'>
Nagy Anna
['kiváló', 'elégedett', 'ajánlom']
0.92


In [ ]:
# ─────────────────────────────────────────
# json.dumps() – Python dict → JSON szöveg
# ─────────────────────────────────────────

eredmeny = {
    "modell": "gemini-1.5-flash",
    "feldolgozott": 42,
    "siker": True,
    "hibak": []
}

# Alap konverzió
szoveg = json.dumps(eredmeny)
print("Alap:")
print(szoveg)
print()

# Formázott (olvasható) kimenet
szep = json.dumps(eredmeny, indent=2, ensure_ascii=False)
print("Formázott:")
print(szep)

Alap:
{"modell": "gemini-1.5-flash", "feldolgozott": 42, "siker": true, "hibak": []}

Formázott:
{
  "modell": "gemini-1.5-flash",
  "feldolgozott": 42,
  "siker": true,
  "hibak": []
}


> **`ensure_ascii=False`** – nélküle a magyar ékezetes betűk unicode escape-ként jelennek meg (`\u00e9` helyett `é`). Mindig adjuk meg, ha magyar szövegeket kezelünk!

In [ ]:
# ─────────────────────────────────────────
# json.dump() és json.load() – fájlba írás és olvasás
# ─────────────────────────────────────────

adatok = {
    "kurzus": "PTE Gépi tanulás",
    "ora": 5,
    "tema": "JSON az LLM programozásban",
    "hallgatok": ["Anna", "Béla", "Csaba"],
    "datum": "2025-03-17"
}

# Fájlba írás
with open("kurzus_adatok.json", "w", encoding="utf-8") as f:
    json.dump(adatok, f, indent=2, ensure_ascii=False)

print("Fájl elmentve: kurzus_adatok.json")

# Visszaolvasás
with open("kurzus_adatok.json", "r", encoding="utf-8") as f:
    betoltott = json.load(f)

print("Visszaolvasott téma:", betoltott["tema"])
print("Hallgatók:", betoltott["hallgatok"])

Fájl elmentve: kurzus_adatok.json
Visszaolvasott téma: JSON az LLM programozásban
Hallgatók: ['Anna', 'Béla', 'Csaba']


### Hibakezelés – mi történik hibás JSON esetén?

Az LLM-ek néha **hibás JSON-t** generálnak (pl. extra szöveg a JSON előtt/után, vagy markdown kódblokk jelölők).  
Fontos, hogy ezt megfelelően kezeljük!

In [ ]:
# Hibás JSON kezelése
hibas_szoveg = """
Íme az eredmény:
{"nev": "Teszt", "ertek": 42}
Remélem segített!
"""

# try/except
try:
    adat = json.loads(hibas_szoveg)
    print("Sikeresen értelmezve:", adat)
except json.JSONDecodeError as e:
    print(f"JSON hiba: {e}")
    print("Megpróbáljuk kinyerni a JSON részt...")

    # Általánosan alkalmazható mentőöv
    eleje = hibas_szoveg.find("{")
    vege  = hibas_szoveg.rfind("}") + 1
    if eleje != -1 and vege > eleje:
        tiszta = hibas_szoveg[eleje:vege]
        adat = json.loads(tiszta)
        print("Kinyert JSON:", adat)

JSON hiba: Expecting value: line 2 column 1 (char 1)
Megpróbáljuk kinyerni a JSON részt...
Kinyert JSON: {'nev': 'Teszt', 'ertek': 42}


In [ ]:
import json
import re # Megjegyzés: Bár most nincs használva, érdemes lehet később regexp-re váltani

# Segédfüggvény: LLM válaszból JSON kinyerése (hasznos pattern!)
def json_kinyerese(llm_szoveg: str) -> dict | None:
    """
    LLM szöveges válaszából megpróbálja kinyerni a JSON tartalmat.
    Visszaadja a dict-et, vagy None-t ha nem sikerül.
    """
    # A felesleges whitespace-ek eltávolítása az elejéről/végéről a pontosabb kereséshez
    szoveg = llm_szoveg.strip()

    # 1. Kísérlet: feltételezzük, hogy a modell "tiszta" JSON-t adott vissza
    # Ez a leggyorsabb eset, ha az LLM-et megfelelően instrukciókkal (system prompt) láttuk el.
    try:
        return json.loads(szoveg)
    except json.JSONDecodeError:
        pass # Ha nem sikerült, folytatjuk a bonyolultabb módszerekkel

    # 2. Kísérlet: Markdown kódblokk keresése (```json ... ```)
    # Az LLM-ek előszeretettel használják a formázást a válasz kiemelésére.
    if "```json" in szoveg:
        eleje = szoveg.find("```json") + 7 # A "```json" hossza 7 karakter, ezt átugorjuk
        vege  = szoveg.find("```", eleje)  # Keressük a blokk lezárását
        if vege != -1: #Ha nem találjuk a blokk végét -1-et kapunk válaszkén
            try:
                # Kivágjuk a blokk tartalmát és megpróbáljuk parsolni
                return json.loads(szoveg[eleje:vege].strip())
            except json.JSONDecodeError:
                pass # Ha a blokkon belül még mindig rossz a formátum, továbbmegyünk

    # 3. Kísérlet: Nyers karakter alapú kinyerés (Brute force keresés)
    # Ha a modell nem használt kódblokkot, de a szöveg tartalmaz egy JSON objektumot.
    # Megkeressük az első '{' és az utolsó '}' pozícióját.
    eleje = szoveg.find("{")
    vege  = szoveg.rfind("}") + 1

    # Ellenőrizzük, hogy érvényes-e az intervallum
    if eleje != -1 and vege > eleje:
        try:
            return json.loads(szoveg[eleje:vege])
        except json.JSONDecodeError:
            pass # Ha így sem sikerül, akkor a szöveg nem tartalmaz érvényes JSON-t

    # Hibakezelés: Amennyiben egyik módszer sem járt sikerrel
    print("FIGYELEM: Nem sikerült JSON-t kinyerni.")
    return None

---
# 4. Beágyazott JSON struktúrák navigálása

Az LLM-ek által visszaadott JSON-ok gyakran **összetett, több szintű** struktúrák.  
Nézzük meg, hogyan navigálunk bennük!

In [ ]:
# Összetett, beágyazott JSON – tipikus LLM kimenet
szamla = {
    "szamla_szam": "2024-042",
    "kelt": "2024-03-15",
    "szallito": {
        "nev": "Pécs Tech Kft.",
        "cim": {
            "varos": "Pécs",
            "utca": "Ifjúság útja 6.",
            "iranyitoszam": "7624"
        },
        "adoszam": "12345678-2-02"
    },
    "tetelek": [
        {"megnevezes": "Laptop Dell XPS", "db": 1, "egysegar": 450000, "afa": 27},
        {"megnevezes": "USB Hub",          "db": 2, "egysegar":   8500, "afa": 27},
        {"megnevezes": "HDMI kábel",       "db": 3, "egysegar":   2900, "afa": 27}
    ],
    "fizetendo": 587640,
    "fizetve": False
}

# Egyszerű mezők elérése
print("Számlaszám:", szamla["szamla_szam"])
print("Szállító:",   szamla["szallito"]["nev"])
print("Város:",      szamla["szallito"]["cim"]["varos"])  # több szint!

# Tömb bejárása
print("\nTételek:")
for i, t in enumerate(szamla["tetelek"], 1):
    osszeg = t["db"] * t["egysegar"]
    print(f"  {i}. {t['megnevezes']} – {t['db']} db × {t['egysegar']:,} Ft = {osszeg:,} Ft")

print(f"\nFizetendő: {szamla['fizetendo']:,} Ft")
print(f"Fizetve: {'Igen' if szamla['fizetve'] else 'Nem'}")

Számlaszám: 2024-042
Szállító: Pécs Tech Kft.
Város: Pécs

Tételek:
  1. Laptop Dell XPS – 1 db × 450,000 Ft = 450,000 Ft
  2. USB Hub – 2 db × 8,500 Ft = 17,000 Ft
  3. HDMI kábel – 3 db × 2,900 Ft = 8,700 Ft

Fizetendő: 587,640 Ft
Fizetve: Nem


In [ ]:
# .get() – biztonságos elérés, ha a kulcs esetleg hiányzik
# (Az LLM-ek néha "hallucinálnak" vagy kihagynak mezőket a válaszból,
# ezért a közvetlen elérés (adat["kulcs"]) gyakran alkalmazás-összeomlást okozna.)

adat = {
    "nev": "Teszt termék",
    "ar": 5000
    # "leiras" és "kategoria" szándékosan hiányzik a példából
}

# ❌ Hibás (kockázatos) módszer:
# print(adat["leiras"])
# Ha a "leiras" kulcs hiányzik, a Python KeyError-t dob, ami leállítja a programot.

# ✅ Helyes módszer: .get(kulcs, alapértelmezett_érték)
# Ha a kulcs megvan, azt adja vissza. Ha nincs, akkor a megadott alapértéket.

# A "nev" kulcs biztosan megvan, de így is biztonságos
nev       = adat.get("nev", "Ismeretlen")

# A "leiras" hiányzik, így a "Nincs leírás" stringet kapjuk vissza
leiras    = adat.get("leiras", "Nincs leírás")

# Ha explicit None-t szeretnénk alapértelmezésnek, azt is megtehetjük
kategoria = adat.get("kategoria", None)

print(f"Név: {nev}")
print(f"Leírás: {leiras}")
print(f"Kategória: {kategoria}")

Név: Teszt termék
Leírás: Nincs leírás
Kategória: None


---
# 5. JSON az LLM programozásban

Az előző gyakorlaton manuálisan másoltuk az LLM által generált JSON-t.  
Most meglátjuk, hogyan kérhetünk **garantáltan valid JSON kimenetet** a Gemini API-tól.

## 5.1 Miért annyira fontos a JSON az LLM-eknél?

Az LLM-ek alapértelmezetten **természetes szöveget** adnak vissza. Ha egy alkalmazásba akarjuk integrálni az LLM kimenetét, azt **gépileg feldolgozható formátumra** kell hozni.

```
Nyers adat (szöveg / kép)
        ↓
   LLM prompt
        ↓
  JSON kimenet  ← ezt akarjuk!
        ↓
  json.loads()
        ↓
  Python dict
        ↓
  Adatbázis / API / Excel / Riport
```

**Két módszer JSON kinyerésére LLM-ből:**
1. **Prompt-alapú** – a promptban kérjük a JSON-t (egyszerű, de nem 100%-os)
2. **API séma-alapú** – a Gemini API-nak megadjuk a pontos JSON sémát (megbízható)

In [ ]:
# Szükséges telepítés (Google Colab)
!pip install google-generativeai --quiet

In [ ]:
import google.generativeai as genai
from google.colab import userdata
import json

# API kulcs betöltése
try:
    api_key = userdata.get("GEMINI_API_KEY")
    print("API kulcs sikeresen betöltve.")
except Exception:
    import getpass
    api_key = getpass.getpass("Add meg a Gemini API kulcsot: ")

genai.configure(api_key=api_key)
model = genai.GenerativeModel("gemini-2.5-flash")
print("Modell inicializálva: gemini-2.5-flash")

/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


API kulcs sikeresen betöltve.
Modell inicializálva: gemini-2.5-flash


## 5.2 Prompt-alapú JSON kérés

A legegyszerűbb módszer: a promptban **expliciten kérjük** a JSON kimenetet.

In [ ]:
def elemzd_szoveget(szoveg: str) -> dict: # A függvény típusjelzése (-> dict) egyértelművé teszi a fejlesztők számára, hogy a hívás után egy Python szótárral dolgozhatnak tovább.
    """Szöveg elemzése – JSON kimenet prompttal kérve."""

    # Itt használjuk a "Few-shot" vagy "Constrained Output" technikát.
    # Fontos, hogy az LLM-et arra utasítjuk, hogy NE írjon körbe semmit ("semmi mást ne írj"),
    # csak a JSON-t adja vissza. Ez a "System Prompt" vagy "Instruction" rész.
    prompt = f"""Elemezd az alábbi szöveget és add vissza az eredményt PONTOSAN ebben a JSON
formátumban, semmi mást ne írj:

{{
  "erzelem": "pozitív" | "semleges" | "negatív",
  "pontszam": 0.0,
  "fo_tema": "a szöveg fő témája 1-2 szóban",
  "kulcsszavak": ["szó1", "szó2", "szó3"]
}}

Szöveg: {szoveg}"""

    # A modell meghívása (ez a konkrét API hívás a Gemini felé)
    valasz = model.generate_content(prompt)

    # Mivel az LLM néha mégis ír "zaj" szöveget (pl. "Természetesen, itt a válasz:"),
    # meghívjuk a korábban definiált segédfüggvényt, ami kitisztítja a kapott text-et.
    return json_kinyerese(valasz.text.strip())

# Tesztelés
teszt = "A Pécs Tech konferencián bemutatott AI termék lenyűgöző volt, teljesen új perspektívákat nyitott meg."
er = elemzd_szoveget(teszt)

# Ha a kinyerés sikeres volt (nem None), akkor szépen formázva kiírjuk a szótárat
if er:
    # json.dumps: a dict-ből újra olvasható stringet csinál,
    # indent=2: szép tagolás, ensure_ascii=False: az ékezetes karakterek megmaradnak
    print(json.dumps(er, indent=2, ensure_ascii=False))

{
  "erzelem": "pozitív",
  "pontszam": 0.9,
  "fo_tema": "AI termék",
  "kulcsszavak": [
    "AI termék",
    "konferencia",
    "lenyűgöző"
  ]
}


## 5.3 API séma-alapú JSON kérés (ajánlott módszer!)

A Gemini API `response_mime_type` és `response_schema` paramétereivel **garantáltan valid JSON-t** kapunk.  
Ez megbízhatóbb, mint a prompt-alapú módszer!

In [ ]:
import json
import google.generativeai as genai
from google.generativeai.types import GenerationConfig
import typing_extensions as typing

# 1. Definiáljuk a várt JSON struktúrát (Schema)
# Ez garantálja, hogy a modell pontosan ezeket a kulcsokat használja.
class ElemzesSchema(typing.TypedDict):
    erzelem: str
    pontszam: float
    fo_tema: str
    kulcsszavak: list[str]

def elemzd_json_api(szoveg: str) -> dict:
    """Szöveg elemzése – Gemini API strukturált JSON kimenettel."""

    prompt = f"""Elemezd az alábbi szöveget:
    - Határozd meg az érzelmi tónust (pozitív / semleges / negatív)
    - Adj egy 0.0-1.0 közötti pontszámot (1.0 = legerősebb érzelem)
    - Határozd meg a fő témát (1-2 szó)
    - Nevezz meg 3 kulcsszót

    Szöveg: {szoveg}"""

    # A konfigurációban adjuk meg a MIME típust és a sémát is
    config = GenerationConfig(
        response_mime_type="application/json",
        response_schema=ElemzesSchema
    )

    valasz = model.generate_content(
        prompt,
        generation_config=config
    )

    # Mivel garantált a struktúra, közvetlenül parse-olhatjuk
    return json.loads(valasz.text)

# Tesztelés
szovegek = [
    "Teljesen elégedetlen vagyok a szolgáltatással, soha többé nem ajánlom.",
    "A meetingre holnap 10 órakor kerül sor a tárgyalóban.",
    "Az új mesterséges intelligencia rendszer elképesztő eredményeket produkál!"
]

print("Elemzések eredményei:\n" + "-"*30)

for sz in szovegek:
    er = elemzd_json_api(sz)

    # Mivel a séma rögzített, ezek a kulcsok garantáltan léteznek:
    print(f"[{er['erzelem'].upper()}] Pont: {er['pontszam']} | Téma: {er['fo_tema']}")
    print(f"Kulcsszavak: {', '.join(er['kulcsszavak'])}")
    print(f"Eredeti: {sz[:50]}...")
    print()

Elemzések eredményei:
------------------------------
[NEGATÍV] Pont: 0.9 | Téma: Szolgáltatás
Kulcsszavak: elégedetlen, szolgáltatás, nem ajánlom
Eredeti: Teljesen elégedetlen vagyok a szolgáltatással, soh...

[SEMLEGES] Pont: 0.1 | Téma: Megbeszélés
Kulcsszavak: megbeszélés, holnap, tárgyaló
Eredeti: A meetingre holnap 10 órakor kerül sor a tárgyalób...

[POZITÍV] Pont: 0.9 | Téma: Mesterséges intelligencia
Kulcsszavak: mesterséges intelligencia, rendszer, eredmények
Eredeti: Az új mesterséges intelligencia rendszer elképeszt...



In [ ]:
import json
import google.generativeai as genai
from google.generativeai.types import GenerationConfig

# 1. Definiáljuk a JSON sémát szótárként (a megadott formátumban)
elemzes_schema = {
    "type": "object",
    "properties": {
        "erzelem": {"type": "string"},
        "pontszam": {"type": "number"},
        "fo_tema": {"type": "string"},
        "kulcsszavak": {
            "type": "array",
            "items": {"type": "string"}
        }
    },
    "required": ["erzelem", "pontszam", "fo_tema", "kulcsszavak"]
}

def elemzd_json_api(szoveg: str) -> dict:
    """Szöveg elemzése – Gemini API strukturált JSON kimenettel."""

    prompt = f"""Elemezd az alábbi szöveget:
    - Határozd meg az érzelmi tónust (pozitív / semleges / negatív)
    - Adj egy 0.0-1.0 közötti pontszámot (1.0 = legerősebb érzelem)
    - Határozd meg a fő témát (1-2 szó)
    - Nevezz meg 3 kulcsszót

    Szöveg: {szoveg}"""

    # A konfigurációban a 'response_schema' paraméterhez adjuk át a sémát
    config = GenerationConfig(
        response_mime_type="application/json",
        response_schema=elemzes_schema
    )

    valasz = model.generate_content(
        prompt,
        generation_config=config
    )

    return json.loads(valasz.text)

# Tesztelés
# Feltételezzük, hogy a 'model' objektum már inicializálva van:
# model = genai.GenerativeModel('gemini-1.5-flash')

szovegek = [
    "Teljesen elégedetlen vagyok a szolgáltatással, soha többé nem ajánlom.",
    "A meetingre holnap 10 órakor kerül sor a tárgyalóban.",
    "Az új mesterséges intelligencia rendszer elképesztő eredményeket produkál!"
]

print("Elemzések eredményei:\n" + "-"*30)

for sz in szovegek:
    er = elemzd_json_api(sz)

    print(f"[{er['erzelem'].upper()}] Pont: {er['pontszam']} | Téma: {er['fo_tema']}")
    print(f"Kulcsszavak: {', '.join(er['kulcsszavak'])}")
    print(f"Eredeti: {sz[:50]}...")
    print()

Elemzések eredményei:
------------------------------
[NEGATÍV] Pont: 0.9 | Téma: Szolgáltatás elégedetlenség
Kulcsszavak: elégedetlen, szolgáltatás, nem ajánlom
Eredeti: Teljesen elégedetlen vagyok a szolgáltatással, soh...

[SEMLEGES] Pont: 0.1 | Téma: Meeting
Kulcsszavak: meeting, holnap, tárgyaló
Eredeti: A meetingre holnap 10 órakor kerül sor a tárgyalób...

[POZITÍV] Pont: 0.9 | Téma: Mesterséges intelligencia
Kulcsszavak: mesterséges intelligencia, rendszer, eredmények
Eredeti: Az új mesterséges intelligencia rendszer elképeszt...



## 5.4 JSON séma megadása – response_schema (haladó)

A **`response_schema`** paraméterrel pontos adatstruktúrát írunk elő – a modell **kötelezően** ezt a sémát követi.  
Ez a legmegbízhatóbb módszer, ha szigorú adatszerkezetre van szükségünk.

In [ ]:
# JSON séma megadása a Gemini API-nak
szamla_schema = {
    "type": "object",
    "properties": {
        "szamla_szam":  {"type": "string"},
        "kelt":         {"type": "string"},
        "szallito": {
            "type": "object",
            "properties": {
                "nev":     {"type": "string"},
                "cim":     {"type": "string"},
                "adoszam": {"type": "string"}
            }
        },
        "tetelek": {
            "type": "array",
            "items": {
                "type": "object",
                "properties": {
                    "megnevezes": {"type": "string"},
                    "mennyiseg":  {"type": "number"},
                    "egysegar":   {"type": "number"},
                    "osszeg":     {"type": "number"}
                }
            }
        },
        "vegosszeg": {"type": "number"},
        "afa_osszeg": {"type": "number"},
        "fizetendo":  {"type": "number"}
    }
}

def szamla_feldolgozas(ocr_szoveg: str) -> dict:
    """Számla OCR szöveg feldolgozása strict JSON sémával."""
    prompt = f"""Az alábbi szöveg egy OCR-rel beolvasott számlából/nyugtából származik.
Nyerd ki az összes fontos adatot. Ha valamit nem találsz, az érték legyen null.

Szöveg:
{ocr_szoveg}"""

    valasz = model.generate_content(
        prompt,
        generation_config=GenerationConfig(
            response_mime_type="application/json",
            response_schema=szamla_schema    # ← szigorú séma!
        )
    )
    return json.loads(valasz.text)

# Teszt – szimulált OCR kimenet
szimulalt_ocr = """
PÉCS TECH KFT
7624 Pécs, Ifjúság útja 6.
Adószám: 12345678-2-02

Számla: 2024-0042
Kelt: 2024-03-15

1x Dell Laptop      450 000 Ft
2x USB Hub           17 000 Ft
3x HDMI kábel         8 700 Ft

Nettó összeg: 374 567 Ft
ÁFA (27%):    101 073 Ft
Fizetendő:    475 640 Ft
"""

eredmeny = szamla_feldolgozas(szimulalt_ocr)
print(json.dumps(eredmeny, indent=2, ensure_ascii=False))

TooManyRequests: 429 POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-flash:generateContent?%24alt=json%3Benum-encoding%3Dint: You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. 
* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-2.5-flash
Please retry in 31.732705978s.

---
# 6. JSON adatok feldolgozása – Pandas DataFrame

Ha több JSON rekordunk van (pl. tömeges LLM feldolgozás eredménye), érdemes **Pandas DataFrame-be** tölteni az egyszerűbb elemzés érdekében.

In [ ]:
import pandas as pd

# Több LLM elemzés eredménye – tipikus tömeges feldolgozás kimenet
elemzesek = [
    {"szoveg": "Remek termék!",                    "erzelem": "pozitív", "pontszam": 0.95, "tema": "termék"},
    {"szoveg": "Teljesen felesleges.",             "erzelem": "negatív", "pontszam": 0.80, "tema": "vélemény"},
    {"szoveg": "Holnap 10 órakor találkozunk.",    "erzelem": "semleges","pontszam": 0.10, "tema": "időpont"},
    {"szoveg": "Fantasztikus előadás volt!",       "erzelem": "pozitív", "pontszam": 0.98, "tema": "esemény"},
    {"szoveg": "Az időjárás változó lesz.",        "erzelem": "semleges","pontszam": 0.05, "tema": "időjárás"},
]

# List of dicts → DataFrame (egyetlen sor!)
df = pd.DataFrame(elemzesek)
print(df)
print()

# Szűrés
print("Pozitív értékelések:")
print(df[df["erzelem"] == "pozitív"][["szoveg", "pontszam"]])
print()

# Statisztika
print("Átlagos pontszám érzelmenként:")
print(df.groupby("erzelem")["pontszam"].mean().round(2))

                          szoveg   erzelem  pontszam      tema
0                  Remek termék!   pozitív      0.95    termék
1           Teljesen felesleges.   negatív      0.80  vélemény
2  Holnap 10 órakor találkozunk.  semleges      0.10   időpont
3     Fantasztikus előadás volt!   pozitív      0.98   esemény
4      Az időjárás változó lesz.  semleges      0.05  időjárás

Pozitív értékelések:
                       szoveg  pontszam
0               Remek termék!      0.95
3  Fantasztikus előadás volt!      0.98

Átlagos pontszám érzelmenként:
erzelem
negatív     0.80
pozitív     0.96
semleges    0.08
Name: pontszam, dtype: float64


In [ ]:
# DataFrame mentése és visszatöltése JSON-ból
df.to_json("elemzesek.json", orient="records", force_ascii=False, indent=2)
print("DataFrame elmentve JSON-ba.")

df2 = pd.read_json("elemzesek.json", encoding="utf-8")
print(f"Visszatöltve, sorok: {len(df2)}")
print(df2.head(2))

DataFrame elmentve JSON-ba.
Visszatöltve, sorok: 5
                 szoveg  erzelem  pontszam      tema
0         Remek termék!  pozitív      0.95    termék
1  Teljesen felesleges.  negatív      0.80  vélemény


---
# 7. Feladatok

## 7.1 Feladat – JSON értelmezése és feldolgozása

Az alábbi JSON szöveg egy webshop rendelési adatait tartalmazza.  
Olvasd be Python-ba és válaszolj a kérdésekre **kód segítségével**!

In [ ]:
rendeles_json = """
{
  "rendeles_id": "ORD-2024-1042",
  "vevo": {
    "nev": "Horváth Mária",
    "email": "horvath.maria@example.com",
    "varos": "Pécs"
  },
  "tetelek": [
    {"termek": "Okosóra Samsung", "db": 1, "egysegar": 89000},
    {"termek": "Töltőkábel USB-C", "db": 2, "egysegar": 2500},
    {"termek": "Védőtok",          "db": 1, "egysegar": 4500}
  ],
  "szallitas": "Foxpost",
  "szallitasi_dij": 990,
  "kupon": "PECS10",
  "kedvezmeny_szazalek": 10
}
"""

# TODO: Oldd meg az alábbi feladatokat!

# 1. Töltsd be a JSON-t Python dict-be
adat = ...

# 2. Írd ki a vevő nevét és városát

# 3. Számítsd ki a tételek összegét

# 4. Számítsd ki a végső fizetendőt
#    (tételek összege + szállítás) × (1 - kedvezmény%)

# 5. Hány darab terméket rendelt összesen?

## 7.1 Megoldás

---
## 7.2 Feladat – Saját JSON séma tervezése

Tervezzük meg egy **könyvtári nyilvántartó rendszer** JSON adatstruktúráját!

A rendszernek az alábbi adatokat kell tárolnia:
- Könyv adatai: cím, szerző (lehet több is!), ISBN, kiadási év, műfaj
- Kölcsönzési állapot: szabad-e; ha kölcsönzött → ki és mikor vette ki, mikor kell visszahozni
- Értékelések: olvasói értékelések (pontszám 1-5, szöveg, dátum)

Írj egy Python dict-et, ami ezt a struktúrát reprezentálja **2 minta könyvvel**!  
Az egyik könyv legyen szabad, a másik kölcsönzött és legyen rajta 2 értékelés.

In [ ]:
import json

# TODO: Töltsd ki a struktúrát!

konyv1 = {
    "cim": ...,
    "szerzok": [...],
    "isbn": ...,
    # kiadasi_ev, mufaj, kolcsonzes, ertekelesek
}

konyv2 = {
    # ...
}

konyvtar = {
    "nev": "PTE Könyvtár",
    "konyvek": [konyv1, konyv2]
}

# Ellenőrzés: mentsd el JSON fájlba és olvasd vissza
with open("konyvtar.json", "w", encoding="utf-8") as f:
    json.dump(konyvtar, f, indent=2, ensure_ascii=False)

## 7.2 Megoldás

In [ ]:
# JSON séma megadása a Gemini API-nak
konyv_schema = {
    "type": "object",
    "properties": {
        "cim":        {"type": "string"},
        "szerzok":    {
            "type": "array",
            "items": {"type": "string"}
        },
        "isbn":       {"type": "string"},
        "kiadas_eve": {"type": "integer"},
        "mufaj":      {"type": "string"},
        "kolcsonzes": {
            "type": "object",
            "properties": {
                "szabad":            {"type": "boolean"},
                "kolcsonzo":         {"type": "string"},
                "kolcsonzes_datuma": {"type": "string"},
                "vissza_datuma":     {"type": "string"}
            },
            # Megadjuk, mely mezők kötelezőek
            "required": ["szabad"]
        },
        "ertekelesek": {
            "type": "array",
            "items": {
                "type": "object",
                "properties": {
                    "olvaso":  {"type": "string"},
                    "pontszam": {"type": "integer"},
                    "szoveg":   {"type": "string"},
                    "datum":    {"type": "string"}
                },
                "required": ["olvaso", "pontszam", "szoveg", "datum"]
            }
        }
    },
    "required": ["cim", "szerzok", "isbn", "kiadas_eve", "mufaj", "kolcsonzes", "ertekelesek"]
}

def szamla_feldolgozas(ocr_szoveg: str) -> dict:
    """Számla OCR szöveg feldolgozása strict JSON sémával."""
    prompt = f"""Az alábbi szöveg egy OCR-rel beolvasott számlából/nyugtából származik.
Nyerd ki az összes fontos adatot. Ha valamit nem találsz, az érték legyen null.

Szöveg:
{ocr_szoveg}"""

    valasz = model.generate_content(
        prompt,
        generation_config=GenerationConfig(
            response_mime_type="application/json",
            response_schema=konyv_schema    # ← szigorú séma!
        )
    )
    return json.loads(valasz.text)

# Teszt – szimulált OCR kimenet
szimulalt_ocr = "A 'Python mesterfokon' című, 2022-ben megjelent programozás témájú könyv szerzője Luciano Ramalho, ISBN száma: 978-963-123456-0. A könyv jelenleg foglalt, Kis Péter kölcsönözte ki 2025. február 28-án, és 2025. március 28-án kell visszahoznia. Két értékelést kapott: Nagy Anna 5 pontot adott rá ('Kiváló referencia könyv!', 2024-11-10-én), Varga Béla pedig 4 pontot ('Haladóknak ajánlom.', 2025-01-05-én)."

eredmeny = szamla_feldolgozas(szimulalt_ocr)
print(json.dumps(eredmeny, indent=2, ensure_ascii=False))

---
## 7.3 Feladat – LLM + JSON pipeline

Készítsetek egy olyan Python programot a Gemini API segítségével, amely egy szállodai foglalási visszaigazolást dolgoz fel, és egyetlen JSON objektumba rendezi az adatokat.

A séma tervezése:
Definiáljatok egy foglalas_schema nevű szótárat (dict), amely a következő adatokat tartalmazza:

Vendég neve (string)

Érkezés és távozás dátuma (string)

Szobák száma (integer)

Összköltség (number)

Szolgáltatások listája (pl. reggeli, szauna, parkolás) – ez egy array legyen!

## Foglalási adatok

Foglalási kérelem (E-mail a vendégtől)
Tárgy: Szobafoglalási igény – Kovács János – 2026. július

Tisztelt Hotel Naplemente!

Szeretnék szobát foglalni az Önök szállodájában 2026. július 10. és 2026. július 15. közötti időszakra. Összesen 2 darab kétágyas szobára lenne szükségem.

Érdeklődöm, hogy a foglalás ára tartalmazza-e a reggelit, a szaunahasználatot és a parkolást is? Kérem, küldjék meg a visszaigazolást, ha a foglalás rögzíthető.

Köszönettel,
Kovács János
+36 30 123 4567

### Válasz:
Tárgy: Foglalási visszaigazolás - Hotel Naplemente
Kedves Kovács János!

Örömmel értesítjük, hogy szállodai foglalását 2026. július 10. és 2026. július 15. között rögzítettük rendszerünkben.

Az ön által lefoglalt 2 darab kétágyas szoba összesen 150.000 Ft-ba kerül. A csomag tartalmazza a bőséges reggelit, az ingyenes szaunahasználatot, valamint a zárt parkolónk használatát a tartózkodás teljes ideje alatt.

Kérjük, érkezéskor mutassa be ezt az e-mailt a recepción.

Üdvözlettel,
A Hotel Naplemente csapata

In [ ]:
import json
import google.generativeai as genai
from google.generativeai.types import GenerationConfig

foglalas_schema = {

email_szoveg = """
Kedves Kovács János!

Örömmel értesítjük, hogy szállodai foglalását 2026. július 10. és 2026. július 15. között rögzítettük rendszerünkben.

Az ön által lefoglalt 2 darab kétágyas szoba összesen 150000 Ft-ba kerül.
A csomag tartalmazza a bőséges reggelit, az ingyenes szaunahasználatot, valamint a zárt parkolónk használatát a tartózkodás teljes ideje alatt.

Kérjük, érkezéskor mutassa be ezt az e-mailt a recepción.

Üdvözlettel,
A Hotel Naplemente csapata
"""


## 7.3 Megoldás

In [ ]:
import json
import google.generativeai as genai
from google.generativeai.types import GenerationConfig

# 1. A JSON séma definiálása a szállodai adatokhoz
foglalas_schema = {
    "type": "object",
    "properties": {
        "vendeg_neve": {"type": "string"},
        "erkezes": {"type": "string"},
        "tavozas": {"type": "string"},
        "szobak_szama": {"type": "integer"},
        "osszkoltseg": {"type": "number"},
        "szolgaltatasok": {
            "type": "array",
            "items": {"type": "string"}
        }
    },
    "required": ["vendeg_neve", "erkezes", "tavozas", "szobak_szama", "osszkoltseg", "szolgaltatasok"]
}

# 2. A feldolgozandó e-mail szövege
email_szoveg = """
Kedves Kovács János!

Örömmel értesítjük, hogy szállodai foglalását 2026. július 10. és 2026. július 15. között rögzítettük rendszerünkben.

Az ön által lefoglalt 2 darab kétágyas szoba összesen 150000 Ft-ba kerül.
A csomag tartalmazza a bőséges reggelit, az ingyenes szaunahasználatot, valamint a zárt parkolónk használatát a tartózkodás teljes ideje alatt.

Kérjük, érkezéskor mutassa be ezt az e-mailt a recepción.

Üdvözlettel,
A Hotel Naplemente csapata
"""

def kinyer_foglalas_adatokat(szoveg: str) -> dict:
    """Feldolgozza az e-mailt és visszaadja az adatokat JSON formátumban."""

    prompt = f"Nyerd ki az adatokat az alábbi foglalási visszaigazolásból a megadott séma szerint: \n\n{szoveg}"

    config = GenerationConfig(
        response_mime_type="application/json",
        response_schema=foglalas_schema
    )

    # Feltételezzük, hogy a 'model' már inicializálva van
    valasz = model.generate_content(prompt, generation_config=config)
    return json.loads(valasz.text)

# --- Futtatás ---
# model = genai.GenerativeModel('gemini-1.5-flash')
eredmeny = kinyer_foglalas_adatokat(email_szoveg)

# Kiírás a konzolra
print("--- Foglalási adatok ---")
print(f"Vendég: {eredmeny['vendeg_neve']}")
print(f"Időtartam: {eredmeny['erkezes']} - {eredmeny['tavozas']}")
print(f"Szobák száma: {eredmeny['szobak_szama']}")
print(f"Összeg: {eredmeny['osszkoltseg']} Ft")
print(f"Szolgáltatások: {', '.join(eredmeny['szolgaltatasok'])}")

TooManyRequests: 429 POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-flash:generateContent?%24alt=json%3Benum-encoding%3Dint: You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. 
* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-2.5-flash
Please retry in 56.106915987s.

---
# Összefoglalás

Ezen a gyakorlaton megtanultuk a JSON alapjait és LLM-es alkalmazásait:

## Tanulságok

| Téma | Legfontosabb tudnivaló |
|---|---|
| JSON szintaxis | Kulcsok dupla idézőjelben; `true`/`false`/`null` kisbetűsen |
| `json.loads()` | JSON szöveg → Python dict |
| `json.dumps()` | Python dict → JSON szöveg; mindig használj `ensure_ascii=False`-t! |
| Hibakezelés | `try/except json.JSONDecodeError` + `.get()` a biztonságos eléréshez |
| LLM + JSON | `response_mime_type="application/json"` garantált JSON-t ad |
| Séma | `response_schema` pontos struktúrát ír elő |

## A JSON helye az LLM pipeline-ban

```
Nyers adat (szöveg / kép)
        ↓
   LLM prompt
        ↓
  JSON kimenet  ←── response_mime_type / response_schema
        ↓
   json.loads()
        ↓
  Python dict / DataFrame
        ↓
  Adatbázis / API / Riport
```

**Következő óra:** Az LLM API haladó funkciói – function calling, rendszer promptok, kontextus kezelés.